# Spatia Fine-tuning Pipeline

Notebook này đóng gói toàn bộ quy trình training từ đầu đến cuối của dự án Spatia bao gồm:
1. **Download:** Tải data từ tập RealEstate10K test set.
2. **Preprocess:** Xử lý video qua VAE và Text qua T5 để trích xuất embedding.
3. **Training Stage 1:** Huấn luyện ControlNet (đóng băng Main Blocks).
4. **Training Stage 2:** Fine-tune Main Blocks bằng LoRA.

In [1]:
import os
import sys
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW

# Đảm bảo import được các module trong project
if str(Path.cwd()) not in sys.path:
    sys.path.append(str(Path.cwd()))

from configs.config import SpatiaConfig
from models import Spatia
from data.dataset import SpatiaDataset
from training.trainer import train_one_epoch
from utils.checkpoint import save_checkpoint
from pipeline.download import download_metadata, download_videos
from pipeline.preprocess import preprocess_all

# Tự động reload module (tiện lợi khi bạn sửa code ở file .py)
%load_ext autoreload
%autoreload 2

### 1. Configuration & Setup
Bạn có thể thay đổi các tham số `max_videos`, `batch_size` hoặc skip các bước nếu đã chạy trước đó.

In [2]:
class Args:
    raw_dir = "data/raw/realestate"
    proc_dir = "data/processed_test"
    max_videos = 100
    batch_size = 2
    device = "cuda" if torch.cuda.is_available() else "cpu"
    vae_name = "Wan-AI/Wan2.2-T2V-1.3B"
    t5_name = "google/t5-v1_1-xxl"
    height = 480
    width = 640
    workers = 4
    max_retries = 3
    skip_download = True    # Bật thành True nếu đã tải rồi
    skip_preprocess = False  # Bật thành True nếu đã preprocess rồi

args = Args()

cfg = SpatiaConfig()
cfg.batch_size = args.batch_size
cfg.height = args.height
cfg.width = args.width
cfg.save_dir = "checkpoints"

raw_dir = Path(args.raw_dir)
proc_dir = Path(args.proc_dir)
raw_dir.mkdir(parents=True, exist_ok=True)

print(f"Device used: {args.device}")

Device used: cpu


### 2. Download Data
Tải metadata và video clip tương ứng.

In [3]:
video_list = []
if not args.skip_download:
    print("Starting download...")
    meta_dir = download_metadata(raw_dir)
    video_dir = raw_dir / "videos"
    video_list = download_videos(
        meta_dir, video_dir,
        max_videos=args.max_videos,
        height=args.height,
        workers=args.workers,
        max_retries=args.max_retries,
    )
else:
    print("Skipping download...")
    video_dir = raw_dir / "videos"
    meta_dir = raw_dir / "test"
    if video_dir.exists() and meta_dir.exists():
        for vp in sorted(video_dir.glob("*.mp4")):
            tp = meta_dir / (vp.stem + ".txt")
            if tp.exists():
                video_list.append((vp, tp))

print(f"Total videos to process: {len(video_list)}")

Skipping download...
Total videos to process: 0


### 3. Preprocess Data
Trích xuất features bằng VAE và Text Embeddings bằng T5.

In [4]:
if not args.skip_preprocess:
    print("Starting preprocessing...")
    preprocess_all(
        video_list, proc_dir, cfg, args.device,
        args.vae_name, args.t5_name,
    )
else:
    print("Skipping preprocessing...")

Starting preprocessing...
[Preprocess] All 0 files already done. Skipping.


### 4. Setup Model & Dataloader
Khởi tạo tập dữ liệu, load model và chuẩn bị cho training.

In [ ]:
device = torch.device(args.device)

# Dataset & DataLoader
dataset = SpatiaDataset(cfg, processed_dir=str(proc_dir))
num_workers = min(4, os.cpu_count() or 1)

dataloader = DataLoader(
    dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(num_workers > 0),
)

n_batches = len(dataloader)
print(f"[Train] {len(dataset)} samples | {n_batches} batches/epoch")

# Initialize Model
model = Spatia(cfg).to(device)
n_param = sum(p.numel() for p in model.parameters()) / 1e6
print(f"[Train] Model: {n_param:.1f} M params")

s1 = min(cfg.stage1_iters, n_batches)
s2 = min(cfg.stage2_iters, n_batches)

[SpatiaDataset] DUMMY MODE — 500 random samples
[Train] 500 samples | 250 batches/epoch
[Train] Model: 849.2 M params


: 

### 5. Training - Stage 1: ControlNet
Ở bước này chúng ta đóng băng kiến trúc chính (Main Blocks) và chỉ huấn luyện ControlNet để hướng dẫn việc sinh frame tiếp theo.

In [ ]:
print("\n" + "─" * 52)
print("  Stage 1 — ControlNet blocks (main blocks frozen)")
print(f"  LR={cfg.lr_controlnet} | iters={s1}")
print("─" * 52)

model.freeze_main_blocks()

opt1 = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=cfg.lr_controlnet, weight_decay=cfg.weight_decay,
)
sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt1, T_max=max(s1, 1), eta_min=cfg.lr_controlnet * 0.1
)

loss1 = train_one_epoch(
    model, dataloader, opt1, sch1, cfg,
    device, stage=1, max_iters=s1
)

save_checkpoint(model, opt1, 1, loss1, cfg.save_dir, "spatia_stage1")
print(f"\n✓ Stage 1 complete! Loss : {loss1:.6f}")


────────────────────────────────────────────────────
  Stage 1 — ControlNet blocks (main blocks frozen)
  LR=1e-05 | iters=250
────────────────────────────────────────────────────


### 6. Training - Stage 2: LoRA
Đóng băng ControlNet và tinh chỉnh Main Blocks bằng phương pháp LoRA.

In [ ]:
print("\n" + "─" * 52)
print("  Stage 2 — LoRA fine-tune main blocks (rank=64)")
print(f"  LR={cfg.lr_lora} | iters={s2}")
print("─" * 52)

model.enable_lora()
model = model.to(device)
model.freeze_controlnet()
model.unfreeze_main_blocks()

opt2 = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=cfg.lr_lora, weight_decay=cfg.weight_decay,
)
sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt2, T_max=max(s2, 1), eta_min=cfg.lr_lora * 0.1
)

loss2 = train_one_epoch(
    model, dataloader, opt2, sch2, cfg,
    device, stage=2, max_iters=s2
)

save_checkpoint(model, opt2, 2, loss2, cfg.save_dir, "spatia_final")
print(f"\n✓ Stage 2 complete! Loss : {loss2:.6f}")

### 7. Evaluation & Inference (Metrics)
Sau khi train xong, chúng ta chạy thử Inference (dùng thuật toán Euler giải phương trình ODE của Flow Matching) để sinh ra latent của frame tiếp theo. Sau đó ta tính PSNR trên không gian latent để đánh giá định lượng độ chính xác của model trước và sau khi train.
*Lưu ý: Để tính PSNR/SSIM trên pixel thật (mắt người nhìn thấy được), bạn cần đưa `x_t_pred` qua hàm decode của VAE.*

In [ ]:
import math
import torch.nn.functional as F

def calculate_latent_psnr(pred_latent, target_latent):
    # Tính Mean Squared Error giữa dự đoán và thực tế
    mse = F.mse_loss(pred_latent, target_latent)
    if mse == 0:
        return 100.0
    # Tín hiệu latent thường có variance xung quanh 1.0
    return 10 * math.log10(1.0 / mse.item())

print("Đang chạy Inference sinh Latent (20 steps)...")
model.eval()
with torch.no_grad():
    # Lấy 1 batch bất kỳ từ tập data
    batch = next(iter(dataloader))
    x_P = batch["x_P"].to(device)
    x_R = batch["x_R"].to(device)
    x_S_T = batch["x_S_T"].to(device)
    x_S_P = batch["x_S_P"].to(device)
    text_tokens = batch["text_tokens"].to(device)
    
    B, N_T, C = batch["x_t"].shape
    
    # Flow Matching: Bắt đầu từ Noise thuần túy
    steps = 20
    dt = 1.0 / steps
    x_t_pred = torch.randn(B, N_T, C, device=device)
    
    # Euler ODE Solver
    for step in range(steps):
        t = torch.ones(B, device=device) * (step * dt)
        velocity = model(x_t_pred, x_P, x_R, x_S_T, x_S_P, text_tokens, t)
        x_t_pred = x_t_pred + velocity * dt
        
    target_latent = batch["x_t"].to(device)
    psnr_score = calculate_latent_psnr(x_t_pred, target_latent)
    
    print(f"\n[Evaluation] Latent PSNR Score: {psnr_score:.2f} dB")
    print("Điểm PSNR càng cao chứng tỏ model sinh ra Latent càng giống với Ground Truth.")
